## News collection from LSEG

+ Start LSEG
+ compile this code

----

## Some Imports to start with

In [1]:
import refinitiv.data as rd
from refinitiv.data.content import news
from IPython.display import HTML
import pandas as pd
import numpy as np
from datetime import datetime,timedelta
import time
import warnings
warnings.filterwarnings("ignore")

## Open the data session

The open_session() function creates and open sessions based on the information contained in the refinitiv-data.config.json configuration file. Please edit this file to set the session type and other parameters required for the session you want to open.

In [2]:
rd.open_session()

<refinitiv.data.session.Definition object at 0x129597910 {name='workspace'}>

## Retrieve data

### Get a list of headlines for a particular query

In [3]:
dNow = datetime.now().date()
maxenddate = dNow - timedelta(days = 500) #upto months=15
compNews = pd.DataFrame()
riclist = ['LCOc1', 'HOc1', 'WTCLc1', '.DJI', '.SPX', 'DIS', 'POL', 'CLc1', 'OVX', 'XOM.N', 'SLB.N'] # can also use Peers, Customers, Suppliers, Monitor, Portfolio to build universe

for ric in riclist:
    try:
        cHeadlines = rd.news.get_headlines("R:" + ric + " AND Language:LEN AND Source:RTRS", start= str(dNow), 
                                           end = str(maxenddate), count = 7000)
        cHeadlines['cRIC'] = ric
        if len(compNews):
            compNews = pd.concat([compNews,cHeadlines])
        else:
            compNews = cHeadlines
    except Exception:
        pass
        
compNews

,headline,storyId,sourceCode,cRIC
versionCreated,,,,
2025-04-13 23:25:00.880,U.S. REFINERY FILING - PORT ARTHUR REFINERY ...,urn:newsml:reuters.com:20250413:nAQN2K9Z4J:1,NS:RTRS,LCOc1
2025-04-13 11:23:53.000,"UPDATE 1-Saudi Arabia, US on 'pathway' to civi...",urn:newsml:reuters.com:20250413:nL1N3QR01H:4,NS:RTRS,LCOc1
2025-04-13 09:42:57.000,US and Saudi Arabia to sign agreement on energ...,urn:newsml:reuters.com:20250413:nS8N3OK03R:1,NS:RTRS,LCOc1
2025-04-13 09:24:36.531,RIYADH - U.S. ENERGY SECRETARY SAYS THERE WILL...,urn:newsml:reuters.com:20250413:nS8N3OK03R:7,NS:RTRS,LCOc1
2025-04-13 09:21:36.293,RIYADH - U.S. ENERGY SECRETARY SAYS FURTHER DE...,urn:newsml:reuters.com:20250413:nS8N3OK03R:6,NS:RTRS,LCOc1
...,...,...,...,...
2024-01-19 11:50:01.418,"SLB Q4 ADJUSTED EBITDA USD 2,277 MILLION VS. I...",urn:newsml:reuters.com:20240119:nPlx1RCB1d:1,NS:RTRS,SLB.N
2024-01-19 11:24:05.000,"US STOCKS-Futures climb as chips, megacaps gai...",urn:newsml:reuters.com:20240119:nL4N3E91WJ:6,NS:RTRS,SLB.N
2024-01-17 17:47:12.000,BUZZ-US energy sector falls on demand concerns...,urn:newsml:reuters.com:20240117:nL4N3E7496:1,NS:RTRS,SLB.N


In [4]:
compNews['timeIssued'] = compNews.index

In [6]:
compNews.head(5)

,headline,storyId,sourceCode,cRIC,timeIssued,storyText,urgency
versionCreated,,,,,,,
2025-04-13 23:25:00.880,U.S. REFINERY FILING - PORT ARTHUR REFINERY ...,urn:newsml:reuters.com:20250413:nAQN2K9Z4J:1,NS:RTRS,LCOc1,2025-04-13 23:25:00.880,U.S. REFINERY FILING PORT ARTHUR REFINERY\n\n(...,3
2025-04-13 11:23:53.000,"UPDATE 1-Saudi Arabia, US on 'pathway' to civi...",urn:newsml:reuters.com:20250413:nL1N3QR01H:4,NS:RTRS,LCOc1,2025-04-13 11:23:53.000,"(Adds detail, context and quotes throughout)\n...",3
2025-04-13 09:42:57.000,US and Saudi Arabia to sign agreement on energ...,urn:newsml:reuters.com:20250413:nS8N3OK03R:1,NS:RTRS,LCOc1,2025-04-13 09:42:57.000,"RIYADH, April 13 (Reuters) - The United States...",3
2025-04-13 09:24:36.531,RIYADH - U.S. ENERGY SECRETARY SAYS THERE WILL...,urn:newsml:reuters.com:20250413:nS8N3OK03R:7,NS:RTRS,LCOc1,2025-04-13 09:24:36.531,"RIYADH, April 13 (Reuters) - The United States...",3
2025-04-13 09:21:36.293,RIYADH - U.S. ENERGY SECRETARY SAYS FURTHER DE...,urn:newsml:reuters.com:20250413:nS8N3OK03R:6,NS:RTRS,LCOc1,2025-04-13 09:21:36.293,"RIYADH, April 13 (Reuters) - The United States...",3


### For each news headline get story text and metadata (topic codes, PermIds, RICs & urgency)

In [7]:
baseurl = "/data/news/v1/stories/"
fullcodelist = pd.DataFrame()
compNews['storyText'] = str()
# compNews['q_codes'] = str()
# compNews['versionCreated'] = str()
# compNews['pIDs_mentioned'] = str()
# compNews['RICs_mentioned'] = str()
compNews['urgency'] = str()

for i, uri in enumerate(compNews['storyId']):
    request_definition = rd.delivery.endpoint_request.Definition(
        url = baseurl + uri,
        method = rd.delivery.endpoint_request.RequestMethod.GET
    )
    response = request_definition.get_data()
    #time.sleep(0.01)
    rawr = response.data.raw
    if 'newsItem' in rawr.keys():
        #compNews['storyText'][i] = rawr['newsItem']['contentSet']['inlineData']['$']
        topics = rawr['newsItem']['contentMeta']['subject']
        rics = [x for x in rawr['newsItem']['assert'] if x['_qcode'].startswith("R:")]
#        compNews['q_codes'][i] = [d['_qcode'] for d in topics]
#        compNews['pIDs_mentioned'][i] = [x for x in compNews['q_codes'][i] if x.startswith("P:")]
#        compNews['RICs_mentioned'][i] = [d['_qcode'] for d in rics] 
        compNews['urgency'] = rawr['newsItem']['contentMeta']['urgency']['$'] # 1 = hot, 3 = regular
        print(i)
compNews

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47


KeyboardInterrupt: 

### Close the session

In [10]:
compNews = compNews[['headline', 'cRIC', 'timeIssued']]

In [12]:
compNews.to_csv('/Users/andaks/Columbia/Columbia Fall24/LLMM/2024_2025_raw_headlines', index=False)

To get OIL or Gold price use Refinitiv -> markets -> commodities -> oil -> historical data -> Excel